### Exploration with local Qwen3.5-2B model

In [1]:
from transformers import pipeline

model = "Qwen/Qwen3.5-2B"

generator = pipeline(
    "text-generation",
     model=model,
     max_length=None,
     max_new_tokens=150, 
     do_sample=True,
     return_full_text=False
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [2]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from app.app import load_resources

2026-04-18 00:16:16.061 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 00:16:16.063 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 00:16:16.128 
  command:

    streamlit run /Users/sapolraadnui/miniforge3/envs/dsci-575-project/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-18 00:16:16.128 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 00:16:16.128 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 00:16:16.129 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-18 00:16:16.129 Thread 'MainThread': missing ScriptRunContext

In [3]:
from app.app import load_resources

In [4]:
documents, bm25 = load_resources() # Create documents

2026-04-18 00:16:16.145 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [5]:
from src.semantic import create_faiss_index

vectorstore = create_faiss_index(documents, 10000, reload_index=False)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

In [7]:
SYSTEM_PROMPT = """
    /no_think
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Recommend ONE product using the context.
Do not add additional explanations or repeat the prompt.
Stop after the recommendation.

Return the answer exactly in this format:

Product Title:
Product ASIN:
Product Rating:
Product Review:
Reason for Recommendation: Write 2 natural sentences describing the product’s key benefits using evidence from the review and rating.

END
"""

In [8]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

In [9]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

format_context = RunnableLambda(build_context)
def prompt_builder(inputs):
    return build_prompt(inputs["input"], inputs["context"])

prompt = RunnableLambda(prompt_builder)


rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
query = "Moisturizing shampoo for thick curly hair"

response = rag_chain.invoke(query)

print(response)

<think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Helpful Amazon shopping assistant.
    *   Task: Recommend ONE product based on the provided context.
    *   Question: "Moisturizing shampoo for thick curly hair"
    *   Constraints:
        *   Answer using ONLY the provided context (product reviews + metadata).
        *   Cite the product ASIN when possible.
        *   Do not add additional explanations (beyond the requested format).
        *   Stop after the recommendation.
        *   Specific Format:
            ```
            Product Title:
            Product ASIN:
            Product Rating:
            Product Review:



### Testing online Meta-Llama-3-8B-Instruct model

In [11]:
from dotenv import load_dotenv

In [12]:
from huggingface_hub import InferenceClient
import os

client = InferenceClient(api_key=os.getenv("HUGGINGFACEHUB_API_TOKEN"))

client.chat_completion(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    messages=[{"role": "user", "content": "Hello"}],
    max_tokens=20,
)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Hello. How can I assist you today?', reasoning=None, tool_call_id=None, tool_calls=None), logprobs=None)], created=1776496604, id='2b0c312ee607cf39f8e56dd9561fae3f', model='meta-llama/llama-3-8b-instruct', system_fingerprint='', usage=ChatCompletionOutputUsage(completion_tokens=10, prompt_tokens=36, total_tokens=46, prompt_tokens_details=None, completion_tokens_details=None), object='chat.completion')

In [13]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Meta-Llama-3-8B-Instruct",
    task="text-generation", # Keep this as text-generation for the base
    max_new_tokens=100,
    huggingfacehub_api_token=token,
    provider="auto" #"novita"
    )

llm = ChatHuggingFace(llm=llm_endpoint)

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful Amazon shopping assistant.

    You must answer using ONLY the information in the context.

    - Recommend ONE product.
    - Do NOT use outside knowledge.
    - Do NOT include any extra text.
    - Return ONLY valid JSON.

    Context:
    {context}

    Question:
    {input}

    Return exactly in this format:

    {{
    "product_title": "",
    "product_asin": "",
    "product_rating": "",
    "product_review": "",
    "reason_for_recommendation": ""
    }}
    """
    )

rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke(query)

'{\n  "product_title": "John Frieda Frizz Ease Hairspray Moist.Barrier 12 Ounce (354ml) (3 Pack)",\n  "product_asin": "B00MMHX1FA",\n  "product_rating": "5.0",\n  "product_review": "I have wavy/ curly hair. I’ve found that if I can spray this in my hair when it is partially dry, it actually holds the curls better.",\n  "reason_for_recommendation'

### This is how you would call it from the function

In [14]:
from src.rag_pipeline import RAGPipeline

rag = RAGPipeline()

response = rag.ask("something to keep your face moisturized all day")

print(response)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'product_title': 'Hyaluronic Acid Serum for face - 11% Low molecules Anti-aging Hydrating Booster Serum 100% Pure Hyaluronic Acid', 'product_asin': 'B087D1YQ2H', 'product_rating': '5.0', 'product_review': 'This is one of the better HA products that I’ve used. It’s great for keeping my skin hydrated all day and hiding fine lines.', 'reason_for_recommendation': 'This product is recommended because it keeps the skin hydrated all day. It can be used both in the morning and under makeup for a smooth and refreshed complexion.'}


## Step 3: Hybrid RAG

We extend the RAG pipeline by combining BM25 keyword search with semantic search using FAISS. This allows better handling of both exact keyword queries and semantic queries.

In [15]:
rag = RAGPipeline(top_k=3)

queries = [
    "moisturizing shampoo for thick curly hair",
    "best product for dry skin",
    "something gentle for sensitive skin",
]

for q in queries:
    print("Query:", q)
    print(rag.ask(q))
    print("-" * 50)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: moisturizing shampoo for thick curly hair
{
    "product_title": "Moroccan Argan Oil Shampoo, 8 Fl Oz - Smooths and Repairs - Sulfate Free - Natural - Imported from Morocco by Pure Body Naturals",
    "product_asin": "B012Q9NGE4",
    "product_rating": "4.0",
    "product_review": "So far I am loving this shampoo! My hair is thick and wavy and some shampoos just dry it out like crazy. But this shampoo actually left my hair feeling soft just the shampoo alone. It has a great smell and is sulfate free so I don't have to worry about it leaching the color fromy dyed hair!  My 6 yo daughter has curly
--------------------------------------------------
Query: best product for dry skin
{'product_title': 'OZNaturals Anti Aging 2.5 Retinol Serum, 30 ml (1 Fl Oz)', 'product_asin': 'B01NB1ZBA1', 'product_rating': '5.0', 'product_review': 'The best product for your skin', 'reason_for_recommendation': 'This product has a high rating and a positive review. This suggests it is a well-regarded p

- We can see that the hybrid retrieval results improved when queries contain keywords like "shampoo" and "moisturizing"
- BM25 helps retrieve exact matches than semantic search may miss
- Semantic retrieval helps when queries are more descriptive and when no exact keywords are available.
- The combination leads to more accurate recommendations.

## Step 5: RAG Evaluation

Manual / Qualitative Evaluation for Hybrid RAG Workflow

In [16]:
queries = [
    # Easy (Keyword-based)
    "ultra facial barrier-hydrating cleanser",
    "AM Facial Moisturizing Lotion SPF 30",
    "fit me concealer",
    "cheek heat gel cream blush, face makeup",
    # Medium (Semantic-based)
    "something to keep your face moisturized all day",
    "makeup to cover up pimples",
    "comfortable lotion for harsh weather",
    # Complex
    "best sunscreen for scuba diving in tropical regions",
    "what’s the best hair treatment to prevent hair loss",
    "good cleanser for busy working professionals who do not have time"
]

for q in queries:
    print("Query:", q)
    print(rag.ask(q))
    print("-" * 50)

Query: ultra facial barrier-hydrating cleanser
{'product_title': 'REBONCEL Aqua Rich Hydrating Face Foam Cleanser Gentle Hypoallergenic pH Balance Korean Skin Care Face Wash Cleansing Foam', 'product_asin': 'B09X23VTSQ', 'product_rating': '5.0', 'product_review': 'Gave my face a nice clean feel without drying it out.', 'reason_for_recommendation': 'This product is suitable for hydrating facial skin, as it is labeled as hydrating and has a 5-star rating. It also has a pH balance that makes it gentle and suitable for sensitive skin.'}
--------------------------------------------------
Query: AM Facial Moisturizing Lotion SPF 30
{'product_title': 'Anew Reversalist Complete Renewal Day Lotion SPF 25', 'product_asin': 'B00JND88A4', 'product_rating': '5.0', 'product_review': 'I like it is creamy and goes on my face so soothing.', 'reason_for_recommendation': 'This product is highly rated and has a soothing effect. It is also a day lotion with SPF, which is likely to provide additional protec